In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/xxxxyyyy80008/amex-test-20020706/amex_test_20220706.parquet


In [2]:
import os, glob, time, gc, psutil
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

path = glob.glob("/kaggle/input/**/amex_test_20220706.parquet", recursive=True)[0]
print(path)
print("File size: %.2f GB" % (os.path.getsize(path) / 1024**3))
print("Total RAM: %.1f GB" % (psutil.virtual_memory().total / 1024**3))

pf = pq.ParquetFile(path)
print("Rows:", pf.metadata.num_rows, "| Columns:", pf.metadata.num_columns)

def ram_gb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**3

/kaggle/input/datasets/xxxxyyyy80008/amex-test-20020706/amex_test_20220706.parquet
File size: 5.78 GB
Total RAM: 31.3 GB
Rows: 11363762 | Columns: 190


In [1]:

import os
import time
import threading
import psutil
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

# ==================================================
# 1. Configuration
# ==================================================
path = "/kaggle/input/datasets/xxxxyyyy80008/amex-test-20020706/amex_test_20220706.parquet"
CHUNK_SIZE = 100_000

if not os.path.exists(path):
    raise FileNotFoundError(f"File not found: {path}")

process = psutil.Process(os.getpid())

# ==================================================
# 2. Inspect Parquet schema
# ==================================================
parquet_file = pq.ParquetFile(path)

numeric_columns = [
    field.name
    for field in parquet_file.schema_arrow
    if (
        pa.types.is_integer(field.type)
        or pa.types.is_floating(field.type)
        or pa.types.is_decimal(field.type)
    )
]

if not numeric_columns:
    raise ValueError("No numeric columns found in the Parquet file.")

print("=" * 60)
print("STREAM PROCESSING - PANDAS CHUNKING")
print("=" * 60)
print(f"File size: {os.path.getsize(path) / (1024**3):.3f} GB")
print(f"Total row groups: {parquet_file.num_row_groups}")
print(f"Numeric columns: {len(numeric_columns)}")
print(f"Chunk size: {CHUNK_SIZE:,} rows")

# ==================================================
# 3. RAM monitoring
# ==================================================
peak_ram_bytes = process.memory_info().rss
stop_monitor = threading.Event()

def monitor_ram():
    global peak_ram_bytes

    while not stop_monitor.is_set():
        try:
            current_ram = process.memory_info().rss
            if current_ram > peak_ram_bytes:
                peak_ram_bytes = current_ram
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            break

        stop_monitor.wait(0.05)

monitor_thread = threading.Thread(
    target=monitor_ram,
    daemon=True
)

# ==================================================
# 4. Stream Processing
# ==================================================
total_rows = 0
total_batches = 0
total_sum = pd.Series(0.0, index=numeric_columns)
total_count = pd.Series(0, index=numeric_columns, dtype="int64")

start_time = time.perf_counter()
monitor_thread.start()

try:
    for batch in parquet_file.iter_batches(
        batch_size=CHUNK_SIZE,
        columns=numeric_columns,
        use_threads=True
    ):
        # Process only the current batch
        df = batch.to_pandas()

        batch_sum = df.sum(axis=0, skipna=True)
        batch_count = df.count(axis=0)

        total_sum = total_sum.add(
            batch_sum, fill_value=0
        )
        total_count = total_count.add(
            batch_count, fill_value=0
        )

        total_rows += len(df)
        total_batches += 1

        del df, batch, batch_sum, batch_count

finally:
    stop_monitor.set()
    monitor_thread.join()

execution_time = time.perf_counter() - start_time

# ==================================================
# 5. Calculate final metrics
# ==================================================
means = total_sum.div(total_count.where(total_count != 0))

peak_ram_mb = peak_ram_bytes / (1024 ** 2)
total_system_ram_mb = psutil.virtual_memory().total / (1024 ** 2)
peak_ram_percent = peak_ram_bytes / psutil.virtual_memory().total * 100

rows_per_second = (
    total_rows / execution_time
    if execution_time > 0 else 0
)

# ==================================================
# 6. Print Results
# ==================================================
print("\n" + "=" * 60)
print("STREAM PROCESSING RESULTS")
print("=" * 60)

print(f"Rows processed          : {total_rows:,}")
print(f"Batches processed       : {total_batches:,}")
print(f"Execution Time          : {execution_time:.2f} seconds")
print(f"Execution Speed         : {rows_per_second:,.2f} rows/second")
print(f"Max RAM Utilization     : {peak_ram_mb:.2f} MB")
print(f"Peak RAM (% total RAM)  : {peak_ram_percent:.2f}%")
print(f"System RAM              : {total_system_ram_mb:.2f} MB")

print("\nMean values (first 20 numeric columns):")
print(means.head(20).to_string())

print("\nMissing-value counts are excluded from each column mean.")
print("Note: RAM percentage above is relative to total system RAM.")


STREAM PROCESSING - PANDAS CHUNKING
File size: 5.780 GB
Total row groups: 1
Numeric columns: 186
Chunk size: 100,000 rows

STREAM PROCESSING RESULTS
Rows processed          : 11,363,762
Batches processed       : 114
Execution Time          : 77.51 seconds
Execution Speed         : 146,601.25 rows/second
Max RAM Utilization     : 6911.14 MB
Peak RAM (% total RAM)  : 21.53%
System RAM              : 32099.07 MB

Mean values (first 20 numeric columns):
P_2     0.657689
D_39    0.156275
B_1     0.127950
B_2     0.616763
R_1     0.074896
S_3     0.227402
D_41    0.059153
B_3     0.143398
D_42    0.148086
D_43    0.161900
D_44    0.117879
B_4     0.175267
D_45    0.245250
B_5     0.089216
R_2     0.047206
D_46    0.471566
D_47    0.396265
D_48    0.376425
D_49    0.199272
B_6     0.155461

Missing-value counts are excluded from each column mean.
Note: RAM percentage above is relative to total system RAM.


In [5]:
import os
import time
import resource

import psutil
import pandas as pd
import pyarrow.parquet as pq   # import BEFORE setting the limit

path = "/kaggle/input/datasets/xxxxyyyy80008/amex-test-20020706/amex_test_20220706.parquet"


def ram_gb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**3


print("RAM before: %.2f GB" % ram_gb())

LIMIT_GB = 4
resource.setrlimit(resource.RLIMIT_AS, (LIMIT_GB * 1024**3, LIMIT_GB * 1024**3))

t0 = time.time()
try:
    df = pd.read_parquet(path)
    print("Loaded OK:", df.shape)
except (MemoryError, OSError) as e:
    print("MemoryError!", repr(e))
    print("RAM at failure: %.2f GB" % ram_gb())
    print("Time until failure: %.1f s" % (time.time() - t0))

RAM before: 0.35 GB
MemoryError! MemoryError()
RAM at failure: 0.42 GB
Time until failure: 0.9 s
